In [1]:
import pandas as pd
import numpy as np

In [2]:
df = pd.read_csv('DB/youtube.csv')

print(df.count())
display(df.head())

link           3599
title          3599
description    3599
category       3599
dtype: int64


,link,title,description,category
0,JLZlCZ0,Ep 1| Travelling through North East India | Of...,Tanya Khanijow\r\n671K subscribers\r\nSUBSCRIB...,travel
1,i9E_Blai8vk,Welcome to Bali | Travel Vlog | Priscilla Lee,Priscilla Lee\r\n45.6K subscribers\r\nSUBSCRIB...,travel
2,r284c-q8oY,My Solo Trip to ALASKA | Cruising From Vancouv...,Allison Anderson\r\n588K subscribers\r\nSUBSCR...,travel
3,Qmi-Xwq-ME,Traveling to the Happiest Country in the World!!,Yes Theory\r\n6.65M subscribers\r\nSUBSCRIBE\r...,travel
4,_lcOX55Ef70,Solo in Paro Bhutan | Tiger's Nest visit | Bhu...,Tanya Khanijow\r\n671K subscribers\r\nSUBSCRIB...,travel


In [3]:
df.drop(columns=['link'], inplace=True)

# Analisis Exploratorio

In [4]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3599 entries, 0 to 3598
Data columns (total 3 columns):
 #   Column       Non-Null Count  Dtype 
---  ------       --------------  ----- 
 0   title        3599 non-null   object
 1   description  3599 non-null   object
 2   category     3599 non-null   object
dtypes: object(3)
memory usage: 84.5+ KB


## Limpieza

Podemos ver que no tenemos datos nulos

In [5]:
df_clean = df.copy()
df_clean.drop_duplicates(inplace=True)

In [6]:
df_clean.count()

title          3433
description    3433
category       3433
dtype: int64

In [7]:
df["title"] = df["title"].astype(str).str.strip()
df["description"] = df["description"].astype(str).fillna("").str.strip()

In [8]:
print("Categorías únicas:")
print(df_clean['category'].unique())

Categorías únicas:
['travel' 'food' 'art_music' 'history']


In [9]:
travel_count = df_clean[df_clean['category'] == 'travel'].shape[0]
print("travel:", travel_count)

food_count = df_clean[df_clean['category'] == 'food'].shape[0]
print("food:", food_count)

art_music_count = df_clean[df_clean['category'] == 'art_music'].shape[0]
print("art_music:", art_music_count)

history_count = df_clean[df_clean['category'] == 'history'].shape[0]
print("history:", history_count)

travel: 1127
food: 887
art_music: 831
history: 588


Como podemos ver nuestras categorias no estan balanceadas, siendo mas las de 'travel' que son casi el doble de las de 'history'

Longitud de caracteres por titulo y descripcion

In [10]:
title_lengths = df_clean['title'].str.len()
print("Titulo")
print("Promedio:", title_lengths.mean())
print("Mínimo:", title_lengths.min())
print("Máximo:", title_lengths.max())


Titulo
Promedio: 67.04340227206525
Mínimo: 6
Máximo: 101


In [11]:
title_lengths = df_clean['description'].str.len()
print("Descripción")
print("Promedio:", title_lengths.mean())
print("Mínimo:", title_lengths.min())
print("Máximo:", title_lengths.max())

Descripción
Promedio: 426.7707544421788
Mínimo: 23
Máximo: 5239


# Modelo

In [12]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification

MODEL_NAME = "xlm-roberta-base"   # multilingüe
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
num_labels = 4

LABEL_LIST = ["travel","food","art_music","history"]
label2id = {l:i for i,l in enumerate(LABEL_LIST)}
id2label = {i:l for l,i in label2id.items()}

c:\Program Files\Python\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [13]:
from transformers import logging as hf_logging
def lengths_percentiles(titles, descs, sample=5000):
    idx = np.random.choice(len(titles), size=min(sample, len(titles)), replace=False)
    prev = hf_logging.get_verbosity()
    hf_logging.set_verbosity_error()  # silencia: "Token indices sequence length..."
    toks = tokenizer(list(titles.iloc[idx]), list(descs.iloc[idx]),
                     truncation=False, padding=False)
    hf_logging.set_verbosity(prev)
    lens = [len(x) for x in toks["input_ids"]]
    return int(np.percentile(lens, 95)), int(np.percentile(lens, 98))

p95, p98 = lengths_percentiles(df["title"], df["description"])
if   p95 <= 128: MAX_LEN = 128
elif p95 <= 256: MAX_LEN = 256
elif p95 <= 384: MAX_LEN = 384
else:            MAX_LEN = 512  # tope XLM-R
print(f"p95: {p95}  p98: {p98}  MAX_LEN: {MAX_LEN}")

p95: 313  p98: 484  MAX_LEN: 384


In [14]:
def tokenize_batch(batch):
    return tokenizer(
        batch["title"],
        batch["description"],
        truncation="only_second",
        max_length=MAX_LEN,
        padding=False
    )

## Separar (train/val/test)

In [15]:
from sklearn.model_selection import train_test_split

LABEL_LIST = ["travel","food","art_music","history"]
label2id = {l:i for i,l in enumerate(LABEL_LIST)}
id2label = {i:l for l,i in label2id.items()}

df = df[df["category"].isin(LABEL_LIST)].copy()
df["label_id"] = df["category"].map(label2id)

train_df, test_df = train_test_split(
    df, test_size=0.10, stratify=df["label_id"], random_state=42
)
train_df, val_df = train_test_split(
    train_df, test_size=0.10/0.90, stratify=train_df["label_id"], random_state=42
)

for name, part in [("Train", train_df), ("Val", val_df), ("Test", test_df)]:
    print(name, "->", len(part), "filas")
    print(part["category"].value_counts(normalize=True).round(3), "\n")


Train -> 2879 filas
category
travel       0.321
art_music    0.263
food         0.251
history      0.165
Name: proportion, dtype: float64 

Val -> 360 filas
category
travel       0.322
art_music    0.264
food         0.250
history      0.164
Name: proportion, dtype: float64 

Test -> 360 filas
category
travel       0.322
art_music    0.264
food         0.250
history      0.164
Name: proportion, dtype: float64 



In [16]:
from datasets import Dataset, DatasetDict

train_ds = Dataset.from_pandas(train_df[["title","description","label_id"]])
val_ds   = Dataset.from_pandas(val_df[["title","description","label_id"]])
test_ds  = Dataset.from_pandas(test_df[["title","description","label_id"]])

ds = DatasetDict({
    "train": train_ds.map(tokenize_batch, batched=True),
    "validation": val_ds.map(tokenize_batch, batched=True),
    "test": test_ds.map(tokenize_batch, batched=True),
})

cols = ["input_ids","attention_mask"]   # RoBERTa: no token_type_ids
ds = ds.remove_columns([c for c in ds["train"].column_names if c not in cols + ["label_id"]])
ds = ds.rename_column("label_id", "labels")
ds.set_format(type="torch", columns=cols+["labels"])


Map: 100%|██████████| 360/360 [00:00<00:00, 6978.84 examples/s]
